# 03 — Expected loss & portfolio concentration
Narrative around `sql/06_marts.sql` and `app/sim_core.py`. Where is risk concentrated,
and what does the policy trade-off look like?

Prereq: `python run_pipeline.py all` has been run.

In [1]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
if not (ROOT / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
np.seterr(all="ignore")
from src.config import load
from src.db import connect
from app.sim_core import approved_metrics, profit_curve, best_policy, money

cfg = load()
con = connect(cfg["paths"]["duckdb"], read_only=True)

In [2]:
el = con.execute("SELECT * FROM mart_loan_el").fetchdf()
print(f"{len(el):,} loans")
el[["pd_hat", "ead", "expected_loss"]].describe().round(2)

640,919 loans


,pd_hat,ead,expected_loss
count,640919.00,640919.00,640919.00
mean,0.13,12726.92,729.35
std,0.07,7882.09,594.46
min,0.00,1000.00,0.00
25%,0.08,7000.00,324.12
50%,0.13,10000.00,565.51
75%,0.18,16850.00,952.42
max,0.64,35000.00,10017.24


## Concentration: share of exposure vs share of expected loss
The memo headline: "segment X is A% of exposure but B% of expected loss".

In [3]:
base = con.execute("SELECT * FROM mart_simulator_base").fetchdf()

def concentration(col):
    g = base.groupby(col).agg(n=("loan_id", "size"), exposure=("loan_amnt", "sum"),
                              el=("expected_loss", "sum"), avg_pd=("pd_hat", "mean"),
                              obs_dr=("default_flag", "mean"))
    g["exposure_share"] = g.exposure / g.exposure.sum()
    g["el_share"] = g.el / g.el.sum()
    g["el_per_exposure"] = (g.el_share / g.exposure_share).round(2)
    return g.sort_values("el_per_exposure", ascending=False).round(3)

concentration("lc_grade")

,n,exposure,el,avg_pd,obs_dr,exposure_share,el_share,el_per_exposure
lc_grade,,,,,,,,
G,546,6.719700e+06,7.362163e+05,0.245,0.41,0.001,0.002,1.91
F,4416,4.369788e+07,4.462107e+06,0.219,0.338,0.005,0.010,1.78
E,22124,2.658408e+08,2.496978e+07,0.205,0.294,0.033,0.053,1.64
D,77437,9.282415e+08,7.710250e+07,0.184,0.239,0.114,0.165,1.45
C,169208,2.045114e+09,1.444736e+08,0.160,0.182,0.251,0.309,1.23
B,221171,2.775407e+09,1.493643e+08,0.124,0.113,0.340,0.320,0.94
A,146017,2.092035e+09,6.634503e+07,0.074,0.054,0.256,0.142,0.55


In [4]:
concentration("purpose")

,n,exposure,el,avg_pd,obs_dr,exposure_share,el_share,el_per_exposure
purpose,,,,,,,,
small_business,6714,9.340000e+07,8.835270e+06,0.207,0.224,0.011,0.019,1.65
educational,1,2.200000e+03,1.836280e+02,0.185,0.0,0.000,0.000,1.46
renewable_energy,431,3.807175e+06,2.745510e+05,0.153,0.197,0.000,0.001,1.26
moving,4491,3.077628e+07,2.180980e+06,0.158,0.196,0.004,0.005,1.24
house,2516,3.203972e+07,2.246424e+06,0.161,0.188,0.004,0.005,1.22
other,33517,2.810280e+08,1.930824e+07,0.154,0.164,0.034,0.041,1.20
vacation,4310,2.430618e+07,1.654287e+06,0.144,0.161,0.003,0.004,1.19
medical,7036,5.214218e+07,3.516033e+06,0.152,0.173,0.006,0.008,1.18
wedding,1174,1.141072e+07,7.656255e+05,0.151,0.127,0.001,0.002,1.17


In [5]:
concentration("vintage_year")

,n,exposure,el,avg_pd,obs_dr,exposure_share,el_share,el_per_exposure
vintage_year,,,,,,,,
2014-01-01,162570,2.046041e+09,1.185865e+08,0.135,0.137,0.251,0.254,1.01
2012-01-01,43470,5.077991e+08,2.920411e+07,0.131,0.136,0.062,0.062,1.00
2015-01-01,283173,3.626461e+09,2.088156e+08,0.134,0.149,0.445,0.447,1.00
2013-01-01,100422,1.272091e+09,7.110732e+07,0.129,0.123,0.156,0.152,0.98
2016-01-01,51284,7.046628e+08,3.973991e+07,0.130,0.148,0.086,0.085,0.98


## Policy trade-off — the same math the simulator uses (`app/sim_core.py`)

In [6]:
SIM = cfg["simulator"]
args = dict(fico_min=SIM["default_min_fico"], cost_of_funds=SIM["cost_of_funds"],
            horizon=SIM["horizon_years"], amort_factor=SIM["amort_factor"],
            servicing_cost=SIM["servicing_cost"])
rows = []
for name, cut in [("Conservative", 0.08), ("Current", 0.15), ("Growth", 0.20)]:
    m = approved_metrics(base, cut, **args)
    rows.append({"policy": name, "pd_cut": cut, **{k: m[k] for k in
                 ("approval_rate", "volume", "exp_default_rate", "exp_loss", "exp_profit")}})
curve = profit_curve(base, args["fico_min"], args["cost_of_funds"], args["horizon"],
                     n_points=120, amort_factor=args["amort_factor"], servicing_cost=args["servicing_cost"])
bp = best_policy(curve)
m = approved_metrics(base, bp["pd_cut"], **args)
rows.append({"policy": "Profit-max", "pd_cut": round(bp["pd_cut"], 3), **{k: m[k] for k in
             ("approval_rate", "volume", "exp_default_rate", "exp_loss", "exp_profit")}})
pd.DataFrame(rows).assign(
    volume=lambda d: d.volume.map(money), exp_loss=lambda d: d.exp_loss.map(money),
    exp_profit=lambda d: d.exp_profit.map(money),
    approval_rate=lambda d: (d.approval_rate * 100).round(1),
    exp_default_rate=lambda d: (d.exp_default_rate * 100).round(2))

,policy,pd_cut,approval_rate,volume,exp_default_rate,exp_loss,exp_profit
0,Conservative,0.080,24.9,$2.4B,5.51,$57.9M,$12.6M
1,Current,0.150,63.3,$5.4B,9.13,$214.3M,$49.5M
2,Growth,0.200,83.0,$6.9B,11.09,$328.9M,$47.5M
3,Profit-max,0.171,72.1,$6.1B,9.97,$262.1M,$51.6M


In [7]:
imax = curve.exp_profit.idxmax()
print(f"profit peaks at PD<{curve.pd_cut[imax]:.3f} / approval {curve.approval_rate[imax]:.1%} "
      f"= {money(curve.exp_profit.max())}")
print(f"at 100% approval profit falls to {money(curve.exp_profit.iloc[-1])} "
      f"-> the curve has a real interior optimum")

profit peaks at PD<0.171 / approval 72.1% = $51.6M
at 100% approval profit falls to $14.1M -> the curve has a real interior optimum


## Scenario stress (a single table — the full layer is deferred, spec §1)
Take the book approved under the CURRENT policy (PD<0.15), then stress: multiply every
PD by a factor and re-price *that same population* (no re-underwriting).

In [8]:
LGD = cfg["expected_loss"]["lgd"]
booked = base[(base.pd_hat < 0.15) & (base.fico_mid >= args["fico_min"])].copy()
stress = []
for f in (1.0, 1.2, 1.5):
    b = booked.copy()
    b["pd_hat"] = (b.pd_hat * f).clip(upper=1.0)
    b["expected_loss"] = b.pd_hat * b.ead * LGD
    # approve everyone in the fixed book (cut-off 1.0), keep all other assumptions
    m = approved_metrics(b, 1.0, fico_min=0, cost_of_funds=args["cost_of_funds"],
                         horizon=args["horizon"], amort_factor=args["amort_factor"],
                         servicing_cost=args["servicing_cost"])
    stress.append({"pd_x": f, "mean_pd": round(b.pd_hat.mean(), 3),
                   "exp_loss": money(m["exp_loss"]), "exp_profit": money(m["exp_profit"])})
pd.DataFrame(stress)

,pd_x,mean_pd,exp_loss,exp_profit
0,1.0,0.091,$214.3M,$49.5M
1,1.2,0.110,$257.1M,$-9.9M
2,1.5,0.137,$321.4M,$-98.9M


In [9]:
con.close()